# 06.LGBM模型开发报告整理

整理 LGBM 各版本结果。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

In [ ]:
# 手动或读取 05 输出结果后汇总
rows = []
for target_type_i in ["y1", "y2"]:
    for model_type in ["不含评分", "含评分_4A", "含评分_4B"]:
        if model_type == "不含评分":
            f = OUTPUT_DIR / f"5.模型效果_LGBM_不含评分_{customer_type}_{target_type_i}.csv"
        else:
            tag = model_type.split("_")[-1].replace("4", "")
            f = OUTPUT_DIR / f"5.模型效果_LGBM_含评分_4{tag}_{customer_type}_{target_type_i}.csv"

        if f.exists():
            r = pd.read_csv(f)
            row = {"建模标签": target_type_i, "模型版本": model_type}
            for ex, prefix in [("train", "train"), ("valid", "valid"), ("oot", "OOT")]:
                tmp = r[r["example"].str.lower() == ex]
                if len(tmp) > 0:
                    row[f"{prefix}_KS"] = tmp["ks"].iloc[0]
                    row[f"{prefix}_AUC"] = tmp["auc"].iloc[0]
            rows.append(row)

model_summary = pd.DataFrame(rows)
display(model_summary)

In [ ]:
report_path = OUTPUT_DIR / f"6.LGBM模型开发报告素材_{customer_type}.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    model_summary.to_excel(writer, sheet_name="1_模型效果汇总", index=False)

print("saved:", report_path)